In [ ]:
# -*- coding: utf-8 -*-
"""Untitled26.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/11DM0bM-LnK1n6ELhliNfxTjtV7eAOByj
"""

import pandas as pd
import io
from google.colab import files
from google.colab import data_table
import re

# 時刻のフォーマットを整形する関数
def format_time_only(value):
    if isinstance(value, str) and ' ' in value:
        return value.split(' ')[-1]
    return value

# 1. ファイル一括アップロードボタンを表示
print("結合したいCSVファイルをすべて選択してアップロードしてください。")
uploaded = files.upload()

if not uploaded:
    print("\nファイルがアップロードされませんでした。処理を中断します。")
else:
    print(f"\n{len(uploaded)}個のファイルがアップロードされました。")

    dataframes = []

    # アップロードされたファイルを名前順に処理
    sorted_filenames = sorted(uploaded.keys())

    for filename in sorted_filenames:
        print(f"処理中: {filename}")
        content = uploaded[filename]

        try:
            # --- ここが修正点：cp923 -> cp932 ---
            # 文字化けしにくい'cp932'(Shift-JIS)を先に試す
            df = pd.read_csv(io.BytesIO(content), header=2, encoding='cp932')
        except Exception:
            # 失敗した場合はUTF-8を試す
            df = pd.read_csv(io.BytesIO(content), header=2, encoding='utf-8')

        if not df.empty:
            # 最初のデータ行（サブヘッダー行）を基準に列を特定
            sub_headers = df.iloc[0]

            # 時刻情報を持つ列（サブヘッダーが '時分'）を見つける
            time_columns_original_names = sub_headers[sub_headers == '時分'].index.tolist()

            rename_map = {}
            for col_name in time_columns_original_names:
                df[col_name] = df[col_name].apply(format_time_only)
                base_name = col_name.rsplit('.', 1)[0]
                new_name = f"{base_name}_時分"
                rename_map[col_name] = new_name

            df.rename(columns=rename_map, inplace=True)

            # 不要な「.＋数字」列を特定する（ただし、処理済みの時刻列は除外）
            all_pattern_columns = [col for col in df.columns if re.search(r'\.\d+$', col)]
            columns_to_drop = [col for col in all_pattern_columns if col not in time_columns_original_names]
            df = df.drop(columns=columns_to_drop)

            # サブヘッダーとして使った最初の行自体も削除
            df = df.iloc[1:].reset_index(drop=True)

            # 「年月日」列のフォーマットを統一
            df['年月日'] = pd.to_datetime(df['年月日'], errors='coerce')

            dataframes.append(df)

    if dataframes:
        # すべてのデータフレームを結合
        merged_df = pd.concat(dataframes, ignore_index=True)

        # 重複した列名を削除（最初の列を優先）
        merged_df = merged_df.loc[:, ~merged_df.columns.duplicated(keep='first')]

        # 年月日が空の行は削除
        merged_df.dropna(subset=['年月日'], inplace=True)

        # --- 日付の抜けをチェック ---
        print("\n--- 日付の抜けチェック ---")
        # 年月日を昇順にソート
        merged_df.sort_values(by='年月日', inplace=True)

        # 連続した日付の範囲を作成
        start_date = merged_df['年月日'].min()
        end_date = merged_df['年月日'].max()
        full_date_range = pd.to_datetime(pd.date_range(start=start_date, end=end_date))

        # 存在する日付のセットを作成
        existing_dates = set(pd.to_datetime(merged_df['年月日']))

        # 存在しない日付（抜け）を見つける
        missing_dates = [date.strftime('%Y-%m-%d') for date in full_date_range if date not in existing_dates]

        if not missing_dates:
            print("✅ チェック完了: 日付の抜けはありませんでした。")
        else:
            print(f"⚠️ チェック完了: {len(missing_dates)}件の日付が抜けています。")
            print("抜けている日付リスト:", missing_dates)
        # ------------------------

        # Excelでの表示のために、年月日を文字列に変換
        merged_df['年月日'] = merged_df['年月日'].dt.strftime('%Y-%m-%d')

        # Colabのインタラクティブなテーブル表示を無効にする
        data_table.disable_dataframe_formatter()

        print("\n--- 結合後のデータプレビュー（最終版） ---")
        display(merged_df)

        # 結合ファイルをBOM付きUTF-8で保存
        output_filename = 'merged_data_final.csv'
        merged_df.to_csv(output_filename, index=False, encoding='utf-8-sig')

        print(f"\nすべてのファイルが結合され、'{output_filename}' として保存されました。")
        print("↓ ダウンロードを開始します。")
        files.download(output_filename)
    else:
        print("\n処理できるファイルがなかったため、結合を中止しました。")

結合したいCSVファイルをすべて選択してアップロードしてください。


Saving 2007年7月7日～2004年7月7日.csv to 2007年7月7日～2004年7月7日.csv
Saving 2010年7月8日～2007年7月8日.csv to 2010年7月8日～2007年7月8日.csv

2個のファイルがアップロードされました。
処理中: 2007年7月7日～2004年7月7日.csv
処理中: 2010年7月8日～2007年7月8日.csv

--- 日付の抜けチェック ---
✅ チェック完了: 日付の抜けはありませんでした。

--- 結合後のデータプレビュー（最終版） ---


,年月日,曜日,平均気温(℃),降水量の合計(mm),最高気温(℃),最高気温(℃)_時分,最低気温(℃),最低気温(℃)_時分,平均風速(m/s),平均湿度(％),天気概況(昼：06時～18時),天気概況(夜：18時～翌日06時)
1,2004-07-07,水,29.8,0.0,34.5,15:30,25.7,05:08,1.8,61.0,曇時々晴,晴一時曇
2,2004-07-08,木,30.9,0.0,35.5,15:13,27.4,05:30,2.2,57.0,晴,晴
3,2004-07-09,金,30.4,0.0,36.3,14:28,27.0,04:28,2.1,58.0,晴,曇
4,2004-07-10,土,26.6,20.5,30.6,15:09,23.2,09:23,1.7,71.0,雨後曇、雷を伴う,曇一時雨、雷を伴う
5,2004-07-11,日,26.2,0.0,31.6,15:09,22.9,05:05,2.0,62.0,晴一時曇,晴後曇
...,...,...,...,...,...,...,...,...,...,...,...,...
2190,2010-07-04,日,26.2,0.0,30.1,13:42,23.4,0:00,2.7,73.0,曇後一時晴,曇時々晴
2191,2010-07-05,月,27.1,0.0,32.8,15:52,22.2,04:59,1.7,64.0,曇時々晴,曇
2192,2010-07-06,火,26.9,0.0,31.9,16:42,24.5,23:34,1.9,67.0,曇,曇
2193,2010-07-07,水,25.4,43.5,31.6,13:01,22.1,14:57,1.6,72.0,薄曇後時々大雨、雷を伴う,曇一時晴



すべてのファイルが結合され、'merged_data_final.csv' として保存されました。
↓ ダウンロードを開始します。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>